In [101]:
import pandas as pd
import plotly.express as px

df1 = pd.read_csv("D:/Python/libraries/orders_raw.csv")
df2 = pd.read_csv("D:/Python/libraries/products_info_raw.csv")


# Data Integration - Mergeovanje

In [102]:
print(df1["Product_ID"].unique()) # Vidimo da ima jedna koja je mali slovima napisano, uradicemo mapping

# df_orders["Product_ID"] = df_orders["Product_ID"].str.upper() - Opsirniji i laksi nacin
df1["Product_ID"] = df1["Product_ID"].replace({"prod_01": "PROD-01"}) # - Moguci nacin

df2["Kat"] = df2["Kat"].replace({"periferija": "Periferija"}).astype("category")

print(df1["Product_ID"].unique()) # Uspelo je! Imamo PROD-1, PROD-2. PROD-3, PROD-4, PROD-5 i PROD-99 koji izbacujemo sa merge

df_final = pd.merge(df1, df2, left_on="Product_ID", right_on="PID", how="inner") # left_on i right_on jer kolone nisu istog naziva

df_final.drop(columns=["PID"], inplace=True)

print(df_final["Product_ID"].unique())

['prod_01' 'PROD-01' 'PROD-03' 'PROD-04' 'PROD-02' 'PROD-99' 'PROD-05']
['PROD-01' 'PROD-03' 'PROD-04' 'PROD-02' 'PROD-99' 'PROD-05']
['PROD-01' 'PROD-03' 'PROD-04' 'PROD-02' 'PROD-05']


# Profitaility Calculation (Feature Engineering)

In [103]:
df_final["Neto_Profit"] = ((df_final["Prodajna"] * (1 - df_final["Popust_%"])) * df_final["Kolicina"]).astype(int)
# Pravimo novu kolonu i tjt
print(df_final["Neto_Profit"].head())

0     1919
1    10800
2      108
3       44
4       81
Name: Neto_Profit, dtype: int64


# Cross-Category Analysis

In [104]:
df_grouped = df_final.groupby("Naziv").agg(suma = ("Neto_Profit", "sum")).reset_index()

fig = px.treemap(df_final, 
                 path=["Kat", "Naziv"], 
                 values="Neto_Profit", 
                 color="Neto_Profit", 
                 color_continuous_scale='RdYlGn', 
                 template="plotly_dark" )

fig.update_traces(hovertemplate="<b>Kategorija:</b> %{parent}<br>" +
                  "<b>Stavka:</b> %{label}<br>" +
                  "<b>Ukupan Profit:</b> %{value:,.0f} EUR<br>" +
                  "<extra></extra>")

fig.show()

# Anomaly Detection (Scatter Plot)

In [105]:
df_final["Popust"] = df_final["Popust_%"] * 100

fig = px.scatter(df_final, 
                 x = "Popust",
                 y = "Neto_Profit",
                 color = "Naziv",
                 size = "Kolicina",
                 hover_data=["Naziv", "Kolicina"],
                 template="plotly_dark" )

fig.update_traces(hovertemplate="<b>Popust (%):</b> %{x}%<br>" +
                  "<b>Profit:</b> %{y:,.0f}<br>" +
                  "<b>Proizvod:</b> %{customdata[0]}<br>" +
                  "<b>Kolicina:</b> %{customdata[1]}<br>" +
                  "<extra></extra>")

fig.show()

# Time Resampling (Vremenska Serija)

In [132]:
df_datum_filter = df_final[df_final["Datum_Prodaje"] <= "2026-04-04"]
df_datum_grouped = df_datum_filter.groupby("Datum_Prodaje")["Neto_Profit"].sum().reset_index()
df_datum_grouped["Kumulativni_Profit"] = df_datum_grouped["Neto_Profit"].cumsum()

fig = px.line(df_datum_grouped, 
              x = "Datum_Prodaje", 
              y = "Kumulativni_Profit", 
              labels={"Datum_Prodaje": "Datum", "Kumulativni_Profit": "Profit"},
              template="plotly_dark")

fig.update_xaxes(dtick="M1", 
                 tickformat="%b %Y", 
                 range=["2025-01-01", "2026-04-04"], 
                 tickangle=30,
                 title_font=dict(size=16, color="#FFF83B"))

fig.update_yaxes(dtick=100000,
                 range=[0, 500000],
                 tickformat=",.0f",
                 title_font=dict(size=16, color="#FFF83B"))

fig.update_traces(hovertemplate="<b>Datum</b>: %{x|%d. %B %Y}<br>" +
                                "<b>Profit</b>: %{y:,.0f} <br>" +
                                "<extra></extra>",
                  line_color="yellow")

fig.update_layout(title=dict(text="Kumulativni profit tokom vremena", x=0.5, xanchor="center", font=dict(size=20, color="#FFF83B")))

fig.show()